# Study Jam ML — Chapter 3 (Challenge)

---

## Beat the Baseline + Save & Use Your Model

**Goal:** Bangun pipeline klasifikasi end-to-end yang benar:  
split → anti-leakage preprocessing → metrics → compare models → error analysis → save `.joblib` → predict test.

### Dataset
- `ps_train.csv` (train + target `satisfaction`)
- `ps_test.csv` (test tanpa label untuk prediksi)

> Kerjakan berurutan. Isi semua bagian bertanda **`# TODO`**.

In [ ]:
# Imports
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score, classification_report
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

---
# Part 1: Download + Load + Split (Stratify)

**Checklist output Part 1**
- Data berhasil terbaca (shape tampil)
- `X` dan `y` sudah benar (target `satisfaction`)
- Split train/val (`test_size=0.2`, `random_state=42`, `stratify=y`)
- Rasio kelas train vs val terlihat mirip

In [ ]:
# Download dataset (do not edit)
!wget -q -O ps_train.csv https://raw.githubusercontent.com/maulnite/Study-Jam-ML-3/main/dataset/passenger_satisfaction/ps_train.csv
!wget -q -O ps_test.csv  https://raw.githubusercontent.com/maulnite/Study-Jam-ML-3/main/dataset/passenger_satisfaction/ps_test.csv

print("Files:", [p.name for p in Path('.').glob('ps_*.csv')])

Files: ['ps_test.csv', 'ps_test_predictions.csv', 'ps_train.csv']


In [ ]:
# Load train
df = pd.read_csv("ps_train.csv")
print("Train shape:", df.shape)
display(df.head())

Train shape: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,3,24026,Female,Loyal Customer,25,Business travel,Business,562,2,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,4,119299,Male,Loyal Customer,61,Business travel,Business,214,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


## Part 1.1 — Define target (y) and features (X)

**Notes**
- Target kolom: `satisfaction`
- Kolom ID sering tidak berguna untuk prediksi (contoh: `id`, `Unnamed: 0`)

Isi cell berikut (lihat TODO).

In [ ]:
# TODO: set target column name

# TODO: define ID-like columns to drop (if any)

# TODO: create X and y

## Part 1.2 — Sanity checks (wajib tampil)

Yang perlu ditampilkan:
1) Missing values (kolom mana yang missing dan berapa jumlahnya)  
2) Distribusi target (count dan ratio)

In [ ]:
# TODO: show missing values summary (only columns with missing > 0)

# TODO: show target distribution (count + ratio)


## Part 1.3 — Split train/validation (stratify)

Pastikan `stratify=y` dipakai.

In [ ]:
# TODO: split train/validation with stratify


---
# Part 2: Baseline + Metrics

**Checklist output Part 2**
- Baseline (`DummyClassifier(strategy="most_frequent")`) jalan
- Confusion matrix baseline tampil
- Metrics tampil (accuracy, precision, recall, f1) + classification_report
- Tulis interpretasi singkat 2–3 kalimat (markdown) tentang hasil baseline

## Part 2.1 — Helper: evaluate_clf (do not edit)

Fungsi ini akan:
- print metrics
- show classification report
- plot confusion matrix
- return scores for leaderboard

In [ ]:
def evaluate_clf(y_true, y_pred, title="Model"):
    print(f"=== {title} ===")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="weighted", zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, average="weighted", zero_division=0))
    print("F1       :", f1_score(y_true, y_pred, average="weighted", zero_division=0))
    print("\nReport:\n", classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=np.unique(y_true))
    ConfusionMatrixDisplay(cm, display_labels=np.unique(y_true)).plot(values_format="d")
    plt.title(f"Confusion Matrix — {title}")
    plt.show()

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_w": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_w": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_w": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

## Part 2.2 — Build preprocessing (anti leakage)

**Requirement**
- Numerical: `SimpleImputer(median)` → `StandardScaler()`  
- Categorical: `SimpleImputer(most_frequent)` → `OneHotEncoder(handle_unknown="ignore")`

Isi bertahap per cell (jangan digabung) supaya gampang dicek.

In [ ]:
# TODO: identify numerical and categorical columns from X_train


In [ ]:
# TODO: build numeric pipeline (imputer median + scaler)


In [ ]:
# TODO: build categorical pipeline (imputer most_frequent + onehot ignore unknown)


In [ ]:
# TODO: combine with ColumnTransformer


## Part 2.3 — Baseline pipeline (wajib)

- Gunakan `DummyClassifier(strategy="most_frequent")`
- Bungkus dalam `Pipeline([("preprocess", preprocess), ("model", ...)])`

In [ ]:
# TODO: build and train baseline pipeline


## Interpretasi Baseline (isi sendiri)

Tulis 2–3 kalimat. Contoh pertanyaan pemandu:
- Baseline cenderung memprediksi kelas apa?
- Apa dampaknya ke confusion matrix?
- Metrik mana yang terlihat tinggi/rendah, dan kenapa?

---
# Part 3: Train Models + Compare Fairly

**Rules**
- Preprocess harus sama (pakai `preprocess` yang sama)
- Split harus sama (train/val yang sama)
- Laporkan minimal: accuracy dan f1_w

**Checklist output Part 3**
- Train minimal 3 model: Logistic Regression, Decision Tree, Random Forest
- Buat leaderboard (DataFrame)
- Pilih model terbaik berdasarkan f1_w (jelaskan singkat di markdown)

## Part 3.1 — Define candidate pipelines (isi TODO)

Isi dictionary `models` di bawah.

In [ ]:
# TODO: define candidate models using the SAME preprocess


## Part 3.2 — Train, evaluate, and build leaderboard (isi TODO)

Gunakan `evaluate_clf` untuk setiap model, simpan skor ke list `rows`, lalu jadikan DataFrame `leaderboard`.

In [ ]:
# TODO: train and evaluate models
rows = []

# TODO: create leaderboard sorted by f1_w desc

## Part 3.3 — Select best model (isi TODO)

Simpan nama model terbaik ke `best_model_name`.

In [ ]:
# TODO: choose best model based on leaderboard (highest f1_w)


## Part 3.4 (Optional) — Cross Validation on best model

Jika kamu kerjakan, tampilkan:
- skor tiap fold
- mean score

In [ ]:
# TODO (optional): run CV on best model using StratifiedKFold
# Hint: StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# Hint: cross_val_score(..., scoring="f1_weighted")

# Uncomment and fill if you do CV
# best_pipe_for_cv = models[best_model_name]
# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# cv_scores = cross_val_score(best_pipe_for_cv, X, y, cv=cv, scoring="f1_weighted")
# print("CV scores:", cv_scores)
# print("CV mean :", cv_scores.mean())

## Interpretasi Pemilihan Model (isi sendiri)

Tulis 1–2 kalimat:
- Model terbaiknya apa?
- Kenapa (berdasarkan metrik di leaderboard)?

---
# Part 4: Mini Error Analysis (wajib, ringkas)

**Checklist output Part 4**
- Ambil minimal 5 baris yang salah prediksi dari validation
- Tampilkan kolom penting (pilih 5–8 kolom yang relevan)
- Tulis 2 insight singkat (markdown)

## Part 4.1 — Get predictions on validation (isi TODO)

- Ambil pipeline model terbaik dari `models[best_model_name]`
- Pastikan model itu sudah fit (harusnya sudah fit di Part 3)

In [ ]:
# TODO: get best pipeline and predict on validation


## Part 4.2 — Show 5 wrong examples (isi TODO)

Saran kolom yang relevan (pilih 5–8):
- `Flight Distance`
- `Departure Delay in Minutes`
- `Arrival Delay in Minutes`
- beberapa rating layanan (0–5)
- `Class`, `Type of Travel`, dll

In [ ]:
# TODO: take 5 wrong examples (you may randomize)


# TODO: optionally select subset of columns for readability


## Insight Error Analysis (isi sendiri)

Tulis 2 insight singkat berbasis contoh salah prediksi di atas.
- Insight 1: ...
- Insight 2: ...

---
# Part 5: Save Model (.joblib) + Load + Predict on Test

**Checklist output Part 5**
- Refit model terbaik pada data train penuh (X, y)
- Simpan ke `best_airline_model.joblib`
- Load kembali file itu
- Prediksi `ps_test.csv`
- Simpan hasil ke `ps_test_predictions.csv` dan tampilkan 10 baris pertama

## Part 5.1 — Refit best model on full data (isi TODO)

Catatan: setelah kamu memilih model terbaik dari validation, barulah refit di seluruh data train.

In [ ]:
# TODO: refit best model on full training data (X, y)

# TODO: save

## Part 5.2 — Load model back (isi TODO)

Pastikan file bisa di-load kembali.

In [ ]:
# TODO: load


## Part 5.3 — Predict on test and save csv (isi TODO)

Requirement:
- Load `ps_test.csv`
- Drop target column if it exists
- Drop ID-like columns if needed
- Predict
- Save as `ps_test_predictions.csv`
- If test has `id`, include it in the output

In [ ]:
# TODO: load test


In [ ]:
# TODO: build X_test (drop target if exists, drop id-like cols if exists)

# TODO: predict

# TODO: build output dataframe

# TODO: include id column if exists

# TODO: save

---
# Submission Checklist

Kumpulkan:
1) Notebook `.ipynb` (run tanpa error)  
2) `best_airline_model.joblib`  
3) `ps_test_predictions.csv`  

Di akhir notebook, tambahkan ringkasan singkat:
- Model terbaik + skor validation
- Kenapa memilih model itu
- 2 insight error analysis